# RAG 성능 극대화 실험: Baseline vs Advanced 대조 평가
본 노트북은 이전 실험에서 우수한 성과를 보였던 **3대 골든 청크 스펙** 조합에 대하여, RAG 검색 및 생성 성능을 한 차원 고도화시키는 **RAG 엔지니어링 개선 기법**들을 전격 적용한 뒤 성능 향상 폭을 Ragas 프레임워크로 객관적으로 대조 검증해보는 고급 실습 공간입니다.

## 📊 RAG 고도화 개선 스펙 (Baseline vs Advanced)
| 구분 | Baseline RAG (기존 모델) | Advanced RAG (개선 모델) |
| :--- | :--- | :--- |
| **1단계 검색 (Recall 개선)** | `k=3` (단순 벡터 유사도 탑 3 검색) | **`k=7` (더 넓고 촘촘하게 7개 청크 수집)** |
| **2단계 정렬 (Precision 개선)** | 리랭킹 없음 | **Flashrank Re-ranker 적용 (상위 핵심 3개로 초정밀 정렬 및 압축)** |
| **프롬프트 통제 (Faithfulness 개선)** | 기본 KORA 전문가 프롬프트 | **탈출 방지 조건(근거 부족 시 100% 모른다고 거절)이 주입된 초강력 프롬프트** |
| **생성 파라미터** | GPT-4o, Temperature 0 | GPT-4o, Temperature 0 (고정) |

## 🎯 검증 대상 우수 조합
1. **Chunk 2000 / Overlap 200** (기존 Recall 최고 조합)
2. **Chunk 2500 / Overlap 200** (기존 고성능 조합)
3. **Chunk 2250 / Overlap 250** (기존 Faithfulness 최고 조합)

In [ ]:
# 1. 필수 및 신규 패키지(flashrank) 설치
%pip install -qU langchain langchain-community langchain-core langchain-text-splitters langchain-openai langchain-chroma pypdf chromadb datasets ragas pandas matplotlib flashrank

In [ ]:
# 2. 라이브러리 로드 및 환경변수 설정
import os
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# LangChain & Chroma 관련 패키지
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Ragas 및 Dataset 패키지
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy

# 상위 경로를 탐색하여 .env 환경변수 로드
load_dotenv(dotenv_path="../../.env")

print("✅ 필수 라이브러리 로드 및 환경변수 설정 완료!")

In [ ]:
# 3. 검증 대상 3개 조합에 대한 Chroma DB 구축 (없는 경우 자동 구축)
pdf_path = "../../docs/KORA 규정집.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"❌ PDF 규정집 파일을 찾을 수 없습니다: {pdf_path}")

print("📖 KORA 규정집 PDF 문서 로드 중...")
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"✅ 문서 로드 성공! 총 {len(documents)} 페이지 확보 완료.")

embedding_model = OpenAIEmbeddings(model='text-embedding-3-large')
print("✨ OpenAI Embeddings 모델 로드 완료!")

# 3대 검증 대상 조합 정의 (chunk_size, overlap_size)
target_combinations = [
    (2000, 200),
    (2500, 200),
    (2250, 250)
 ]

print("\n🚀 대상 조합별 벡터 DB 순차 구축 시작...")
for chunk_size, overlap_size in target_combinations:
    persist_dir = f"./chroma_kora_{chunk_size}_overlap_{overlap_size}"
    
    # 폴더가 이미 완벽하게 구축되어 존재할 경우 가볍게 스킵
    if os.path.exists(persist_dir) and len(os.listdir(persist_dir)) > 0:
        print(f" 💾 [Skip] {chunk_size}/{overlap_size} DB가 이미 완벽히 구축되어 빌드를 생략합니다.")
        continue
        
    print(f"\n⚙️ [빌드 가동] Chunk Size: {chunk_size} | Overlap: {overlap_size} ...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap_size)
    document_list = text_splitter.split_documents(documents)
    print(f"  ✂️ 문서를 {len(document_list)}개의 조각으로 쪼갰습니다.")
    
    print(f"  💾 '{persist_dir}' 폴더에 벡터 DB 저장 중...")
    database = Chroma.from_documents(
        documents=document_list,
        embedding=embedding_model,
        collection_name=f"chroma-kora-{chunk_size}-{overlap_size}",
        persist_directory=persist_dir
    )
    print(f"  ✅ {chunk_size}/{overlap_size} 벡터 DB 로컬 저장 성공!")
    time.sleep(1)

print("\n🎉 평가 대상 모든 벡터 DB 세팅이 완료되었습니다!")

In [ ]:
# 4. 평가용 골든 데이터셋 로드
print("--- 1. 평가용 데이터셋 로드 ---")
dataset_path = "../../docs/kora_eval_dataset.json"

try:
    with open(dataset_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    test_questions = [item['question'] for item in eval_data['dataset']]
    ground_truths = [item['ground_truth'] for item in eval_data['dataset']]
    print(f"🎉 총 {len(test_questions)}개의 평가용 테스트 데이터셋이 정상 로드되었습니다!")
except FileNotFoundError:
    print(f"❌ {dataset_path} 파일을 찾을 수 없습니다. 경로를 다시 한 번 확인해주세요.")
    test_questions = []
    ground_truths = []

In [ ]:
# 5. Baseline 및 Advanced RAG 답변 생성 및 데이터 수집
all_evaluation_data = {}

if not test_questions:
    print("❌ 평가 데이터셋이 존재하지 않아 수집을 가동하지 않습니다.")
else:
    target_combinations = [(2000, 200), (2500, 200), (2250, 250)]
    embedding_model = OpenAIEmbeddings(model='text-embedding-3-large')
    llm = ChatOpenAI(model='gpt-4o', temperature=0)
    
    # 1. Baseline 용 기본 프롬프트
    baseline_prompt = ChatPromptTemplate.from_template("""[Identity]
- 당신은 최고의 KORA 규정 전문가입니다.
- [Context]를 참고해서 사용자의 질문에 정확하고 친절하게 답변해주세요.
- 문서에 정보가 없는 경우 솔직하게 모른다고 대답하세요.

[Context]
{context}

Question: {question}
""")
    
    # 2. Advanced 용 탈출 방지 강화 프롬프트 (Faithfulness 극대화)
    advanced_prompt = ChatPromptTemplate.from_template("""[Identity]
- 당신은 철저하게 KORA 규정집의 객관적인 팩트에만 기반하여 답변하는 규정 분석 전문가입니다.

[Rules]
- 아래 제공된 [Context]를 정독하고, 사용자의 질문에 정확하게 답변해주세요.
- **핵심 규칙**: 답변을 작성할 때 반드시 [Context]에 명시된 구체적인 문구, 조항, 수치만 그대로 사용해야 합니다.
- **탈출 방지 조건 (환각 완전 차단)**: 질문에 대답하기 위한 확실하고 명확한 근거가 [Context]에 전혀 없거나 불충분한 경우, 절대로 본인의 지식이나 추측을 섞어 답변하지 마십시오. 그러한 경우에는 반드시 정확하게 다음 문장으로만 일치되게 답변하십시오: "제공된 문서(Context)에서 해당 질문에 대한 관련 정보를 찾을 수 없습니다."

[Context]
{context}

Question: {question}
""")
    
    for chunk_size, overlap_size in target_combinations:
        persist_dir = f"./chroma_kora_{chunk_size}_overlap_{overlap_size}"
        
        if not os.path.exists(persist_dir):
            print(f"  ⚠️ {persist_dir} DB가 존재하지 않아 건너뜁니다.")
            continue
            
        # Chroma DB 연결
        database = Chroma(
            collection_name=f"chroma-kora-{chunk_size}-{overlap_size}",
            persist_directory=persist_dir,
            embedding_function=embedding_model
        )
        
        # =========================================================================
        # [A] Baseline RAG 설정 및 수집 (k=3, 리랭커 X, 기본 프롬프트)
        # =========================================================================
        base_key = f"Baseline ({chunk_size}/{overlap_size})"
        print(f"\n🚀 [수집 가동] {base_key} 구동 중...")
        
        baseline_retriever = database.as_retriever(search_kwargs={'k': 3})
        baseline_chain = (
            {"context": baseline_retriever, "question": RunnablePassthrough()}
            | baseline_prompt
            | llm
            | StrOutputParser()
        )
        
        baseline_data = {"question": [], "contexts": [], "answer": [], "ground_truth": []}
        for i, q in enumerate(test_questions):
            time.sleep(0.3)
            try:
                answer = baseline_chain.invoke(q)
                retrieved_docs = baseline_retriever.invoke(q)
                contexts = [doc.page_content for doc in retrieved_docs]
                
                baseline_data["question"].append(q)
                baseline_data["contexts"].append(contexts)
                baseline_data["answer"].append(answer)
                baseline_data["ground_truth"].append(ground_truths[i])
            except Exception as e:
                print(f"    ⚠️ Baseline 질문 {i+1} 오류: {e}")
                
        all_evaluation_data[base_key] = baseline_data
        print(f"  ✅ {base_key} 완료! (10개 질문)")
        
        # =========================================================================
        # [B] Advanced RAG 설정 및 수집 (k=7 -> Flashrank 리랭킹 -> Top 3 압축, 강화 프롬프트)
        # =========================================================================
        adv_key = f"Advanced ({chunk_size}/{overlap_size})"
        print(f"🚀 [수집 가동] {adv_key} 구동 중...")
        
        # Flashrank Reranker 적용 시도
        try:
            from langchain.retrievers import ContextualCompressionRetriever
            from langchain.retrievers.document_compressors import FlashrankRerank
            
            compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2")
            advanced_retriever = ContextualCompressionRetriever(
                base_compressor=compressor,
                base_retriever=database.as_retriever(search_kwargs={'k': 7}) # 1단계: 7개 청크 검색 (Recall 극대화)
            )
            # Compressor로 압축 정렬 시 기본적으로 점수가 가장 높은 청크들이 상위 정렬되어 LLM에 전달됩니다.
            is_reranked = True
        except Exception as e:
            print(f"  ⚠️ Flashrank 로드 실패로 기본 retriever(k=5)로 긴급 Fallback 작동합니다. 에러: {e}")
            advanced_retriever = database.as_retriever(search_kwargs={'k': 5})
            is_reranked = False
            
        advanced_chain = (
            {"context": advanced_retriever, "question": RunnablePassthrough()}
            | advanced_prompt
            | llm
            | StrOutputParser()
        )
        
        advanced_data = {"question": [], "contexts": [], "answer": [], "ground_truth": []}
        for i, q in enumerate(test_questions):
            time.sleep(0.3)
            try:
                answer = advanced_chain.invoke(q)
                retrieved_docs = advanced_retriever.invoke(q)
                contexts = [doc.page_content for doc in retrieved_docs]
                
                advanced_data["question"].append(q)
                advanced_data["contexts"].append(contexts)
                advanced_data["answer"].append(answer)
                advanced_data["ground_truth"].append(ground_truths[i])
            except Exception as e:
                print(f"    ⚠️ Advanced 질문 {i+1} 오류: {e}")
                
        all_evaluation_data[adv_key] = advanced_data
        print(f"  ✅ {adv_key} 완료! (10개 질문) [리랭커 사용여부: {is_reranked}]")
        
    print("\n🎉 모든 대상 조합 및 시나리오(6개 스펙) RAG 답변 데이터 수집 성공!")

In [ ]:
# 6. Ragas 평가 수행 및 종합 스코어 변환
evaluation_results = {}

if not all_evaluation_data:
    print("❌ 채점 대상 데이터가 준비되지 않아 Ragas 채점을 중단합니다.")
else:
    print("--- 3. 6개 스펙 Ragas 지표 채점 시작 (OpenAI API 채점 가동) ---\n")
    
    # Ragas 평가 모델을 gpt-4o-mini로 명시 세팅하여 평가 속도 및 API 비용 대폭 절감
    eval_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    
    for spec_key, data in all_evaluation_data.items():
        print(f"⚖️ [채점 중] {spec_key} Ragas 평가 수행 중...")
        try:
            # Dataset 생성
            eval_dataset = Dataset.from_dict(data)
            
            # Ragas evaluate 호출 (평가 모델 고정 적용)
            result = evaluate(
                dataset=eval_dataset,
                metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                llm=eval_llm
            )
            
            # 결과 데이터프레임 변환 후 각 지표 평균 추출
            df_details = result.to_pandas()
            target_metrics = ['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']
            existing_metrics = [col for col in target_metrics if col in df_details.columns]
            
            clean_scores = df_details[existing_metrics].mean().to_dict()
            evaluation_results[spec_key] = clean_scores
            print(f"  🎉 {spec_key} 채점 완료! [평균 결과]: {clean_scores}")
        except Exception as e:
            print(f"  ❌ {spec_key} 채점 실패: {e}")
            evaluation_results[spec_key] = {
                'context_precision': float('nan'),
                'context_recall': float('nan'),
                'faithfulness': float('nan'),
                'answer_relevancy': float('nan')
            }
        print("-" * 40)
        time.sleep(2)

    print("\n🏁 6개 스펙 Ragas 대조 채점 루프가 완벽히 종료되었습니다!")

In [ ]:
# 7. 다중 청크 성능 대조 리포트 테이블 및 Baseline vs Advanced 비교 시각화
if not evaluation_results:
    print("❌ 출력할 데이터가 존재하지 않습니다.")
else: 
    report_rows = []
    for spec_key, result in evaluation_results.items():
        # 키 분리 (예: "Baseline (2000/200)" -> Type: Baseline, Specs: 2000/200)
        is_baseline = "Baseline" in spec_key
        rag_type = "Baseline" if is_baseline else "Advanced"
        spec_nums = spec_key.split("(")[1].replace(")", "")
        
        report_rows.append({
            "RAG Type": rag_type,
            "Specs (Size/Overlap)": spec_nums,
            "context_precision": result.get("context_precision", 0.0),
            "context_recall": result.get("context_recall", 0.0),
            "faithfulness": result.get("faithfulness", 0.0),
            "answer_relevancy": result.get("answer_relevancy", 0.0)
        })
        
    df_final = pd.DataFrame(report_rows)
    print("📊 [RAG 고도화 성능 대조 리포트 테이블]")
    display(df_final.sort_values(by=["Specs (Size/Overlap)", "RAG Type"]))
    
    # =========================================================================
    # Baseline vs Advanced 성능 향상 폭 다중 막대 그래프 시각화
    # =========================================================================
    plot_df = df_final.fillna(0.0)
    
    # 지표 리스트 정의
    metrics = ['context_precision', 'context_recall', 'faithfulness']
    specs = plot_df["Specs (Size/Overlap)"].unique()
    
    # 3대 지표에 대해 3개의 서브플롯 차트 그리기
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        
        # 해당 지표에 대한 데이터 피벗 테이블화
        pivot_df = plot_df.pivot(index="Specs (Size/Overlap)", columns="RAG Type", values=metric)
        
        # Baseline vs Advanced 컬럼 배치 고정
        if "Baseline" in pivot_df.columns and "Advanced" in pivot_df.columns:
            pivot_df = pivot_df[["Baseline", "Advanced"]]
            
        pivot_df.plot(kind='bar', ax=ax, color=['#b0c4de', '#4682b4'], width=0.6, edgecolor='black')
        
        ax.set_title(f"{metric.replace('_', ' ').title()}", fontsize=13, fontweight='bold')
        ax.set_xlabel("Specs (Size/Overlap)", fontsize=11)
        ax.set_ylim(0, 1.1)
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        ax.legend(loc='lower left')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        
        # 막대 위에 값 레이블 띄우기
        for p in ax.patches:
            height = p.get_height()
            if height > 0:
                ax.annotate(f"{height:.3f}",
                            xy=(p.get_x() + p.get_width() / 2, height),
                            xytext=(0, 3), # 3포인트 위
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=9, fontweight='semibold')
                
    plt.suptitle("RAG Optimization Effect: Baseline vs Advanced (k=7 + Re-ranker + Prompt Guard)", 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # 대조 차트 이미지 로컬 저장
    report_chart_path = "./rag_advanced_comparison_chart.png"
    plt.savefig(report_chart_path, dpi=300, bbox_inches='tight')
    print(f"\n📈 Baseline vs Advanced 대조 분석 시각화 차트가 '{report_chart_path}'에 저장되었습니다!")
    plt.show()